# Import Library

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import hstack
from collections import defaultdict
from difflib import get_close_matches
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import coo_matrix

# UI using tkinter
import sys
from PyQt5.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout, 
                             QHBoxLayout, QLineEdit, QPushButton, QListWidget, 
                             QLabel, QStackedWidget, QComboBox, QSpinBox, 
                             QDoubleSpinBox, QMessageBox, QListWidgetItem)
from PyQt5.QtCore import Qt
from PyQt5.QtGui import QFont, QPixmap
from PyQt5.QtMultimedia import QMediaPlayer, QMediaContent
from PyQt5.QtCore import QUrl

# Content-Based Filtering

In [2]:
df = pd.read_csv('data/newMusicInfo.csv')
df.head()

,track_id,name,artist,spotify_preview_url,tags,genre,year,duration_ms,danceability,speechiness,tempo
0,TROUVHL128F426C441,Come as You Are,NIRVANA,https://p.scdn.co/mp3-preview/a1c11bb1cb231031...,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, GRUNGE",RNB,1991,218920,0.508,0.0400,120.012
1,TRLNZBD128F935E4D8,Creep,RADIOHEAD,https://p.scdn.co/mp3-preview/e7eb60e9466bc3a2...,"ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...",RNB,2008,238640,0.515,0.0369,91.841
2,TRMZXEW128F9341FD5,The Scientist,COLDPLAY,https://p.scdn.co/mp3-preview/95cb9df1b056d759...,"ROCK, ALTERNATIVE, INDIE, POP, ALTERNATIVE_ROC...",ROCK,2007,311014,0.566,0.0242,146.365
3,TRIODZU128E078F3E2,Under the Bridge,RED HOT CHILI PEPPERS,https://p.scdn.co/mp3-preview/90e41778392f27b6...,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, FUNK",POP,2003,265506,0.554,0.0457,84.275
4,TRYEZGE128F92D04C8,Feel Good Inc.,GORILLAZ,https://p.scdn.co/mp3-preview/7388c425022ced92...,"ROCK, ELECTRONIC, ALTERNATIVE, INDIE, POP, ALT...",RNB,2010,223106,0.809,0.1360,138.556


### Selected only the relevant features

In [3]:
df_selected = df[['track_id', 'name','artist', 'tags', 'genre', 'year', 'danceability', 'speechiness', 'tempo','spotify_preview_url','duration_ms']]
df_selected.head()

,track_id,name,artist,tags,genre,year,danceability,speechiness,tempo,spotify_preview_url,duration_ms
0,TROUVHL128F426C441,Come as You Are,NIRVANA,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, GRUNGE",RNB,1991,0.508,0.0400,120.012,https://p.scdn.co/mp3-preview/a1c11bb1cb231031...,218920
1,TRLNZBD128F935E4D8,Creep,RADIOHEAD,"ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...",RNB,2008,0.515,0.0369,91.841,https://p.scdn.co/mp3-preview/e7eb60e9466bc3a2...,238640
2,TRMZXEW128F9341FD5,The Scientist,COLDPLAY,"ROCK, ALTERNATIVE, INDIE, POP, ALTERNATIVE_ROC...",ROCK,2007,0.566,0.0242,146.365,https://p.scdn.co/mp3-preview/95cb9df1b056d759...,311014
3,TRIODZU128E078F3E2,Under the Bridge,RED HOT CHILI PEPPERS,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, FUNK",POP,2003,0.554,0.0457,84.275,https://p.scdn.co/mp3-preview/90e41778392f27b6...,265506
4,TRYEZGE128F92D04C8,Feel Good Inc.,GORILLAZ,"ROCK, ELECTRONIC, ALTERNATIVE, INDIE, POP, ALT...",RNB,2010,0.809,0.1360,138.556,https://p.scdn.co/mp3-preview/7388c425022ced92...,223106


### Check data type

In [4]:
df_selected.dtypes

track_id                object
name                    object
artist                  object
tags                    object
genre                   object
year                     int64
danceability           float64
speechiness            float64
tempo                  float64
spotify_preview_url     object
duration_ms              int64
dtype: object

### Copy and clean

In [5]:
df_selected = df[['track_id', 'name', 'artist', 'tags', 'genre', 'year', 'danceability', 'speechiness', 'tempo','spotify_preview_url','duration_ms']].copy()
df_selected[['name', 'artist', 'tags', 'genre']] = df_selected[['name', 'artist', 'tags', 'genre']].fillna('')

### Combine text features

In [6]:
df_selected['combined_text'] = df_selected['tags'] + ' ' + df_selected['genre'] + ' ' + df_selected['artist']

### TF-IDF on combined text

In [7]:
# Initialize TF-IDF vectorizer
tfidf = TfidfVectorizer(
    analyzer='word', 
    ngram_range=(1, 2),  # Include unigrams and bigrams
    min_df=1, # Keep words that appear in at least 1 document
    stop_words='english'
)

# Transform the combined text column into TF-IDF features
text_matrix = tfidf.fit_transform(df_selected['combined_text'])

### Normalize numeric features

In [8]:
# Initialize and apply MinMaxScaler to scale all numeric columns to [0, 1]
scaler = MinMaxScaler()
# Apply scaling to numerical features
numeric_matrix = scaler.fit_transform(df_selected[['danceability', 'speechiness', 'tempo', 'year']])

### Combine all features

In [9]:
combined_matrix = hstack([text_matrix, numeric_matrix]).tocsr()

### Fit KNN model

In [10]:
knn_model = NearestNeighbors(metric='cosine', algorithm='brute')
knn_model.fit(combined_matrix)

NearestNeighbors(algorithm='brute', metric='cosine')

### Build name, tag, genre, year and index mapping

In [11]:
# lowercase song names in mapping
# Build the (song name) -> index map
name_to_index = {
    row['name'].lower().strip(): idx
    for idx, row in df_selected.iterrows()
}

In [12]:
# Create a dictionary to map (tag, genre, year) -> list of row indices
tag_genre_year_to_indices = {}

# Loop through each row in the dataframe
for idx, row in df_selected.iterrows():
    # Build a unique key using tag, genre, and year (all lowercase for consistency)
    key = (row['tags'].lower().strip(), row['genre'].lower().strip(), row['year'])
    
    # Insert the key into the dictionary if it doesn't exist, and append the current row index to its list
    tag_genre_year_to_indices.setdefault(key, []).append(idx)

In [13]:
# Create a dictionary to map (genre, year) -> list of row indices
genre_year_to_indices = {}

# Loop through each row in the dataframe
for idx, row in df_selected.iterrows():
    # Build a unique key using genre and year (all lowercase for consistency)
    key = (row['genre'].lower().strip(), row['year'])
    
     # Insert the key into the dictionary if it doesn't exist, and append the current row index to its list
    genre_year_to_indices.setdefault(key, []).append(idx)

In [14]:
genre_to_indices = {}

# Loop through each row in the dataframe
for idx, row in df_selected.iterrows():
    # Build a unique key using genre (all lowercase for consistency)
    key = str(row['genre']).lower().strip()
    
     # Insert the key into the dictionary if it doesn't exist, and append the current row index to its list
    genre_to_indices.setdefault(key, []).append(idx)

### Recommendation function using name

##### convert time ms to min

In [15]:
def ms_to_minutes_seconds(ms):
    try:
        ms = int(ms)
        seconds = ms // 1000
        minutes = seconds // 60
        remaining_seconds = seconds % 60
        return f"{minutes}:{remaining_seconds:02d}"
    except Exception:
        return None

##### Calculation accuracy function

In [16]:
def calculate_accuracy_from_distance(distances):
    # Convert KNN distances into similarity percentages.
    return [(1 - d) * 100 for d in distances]

##### Validation for tempo, danceability and year

In [17]:
def validate_song_name(song_name):
    # Validate that the song exists in the dataset
    song_key = song_name.lower().strip()
    if song_key not in name_to_index:
        return None, f"Song '{song_name}' not found in the dataset."
    return song_key, None

##### Main Recommendation using name Function

In [18]:
def recommend_songs_by_name(song_name, n_recommendations=5):
    # Preprocess the input name (normalize: lowercase + remove spaces)
    song_key = song_name.lower().strip()
    
    # 1 - Check if the song exists in the mapping
    song_key, error = validate_song_name(song_name)
    if error:
        return error
    
    # 2 - Get the index of the input song directly from the dictionary
    idx = name_to_index[song_key] 
    
    # 3 - Find nearest neighbors using the KNN model
    distances, indices = knn_model.kneighbors(
        combined_matrix[idx], n_neighbors=n_recommendations + 1
    )
    
    # 4 - Exclude the first result (it’s always the song itself)
    recommended_indices = indices.flatten()[1:]
    recommended_distances = distances.flatten()[1:]
    
    # 5 - Get details of the input song
    input_song = df_selected.iloc[[idx]][
        ['track_id','name','artist','year','genre','tempo','tags','danceability','spotify_preview_url','duration_ms']
    ]
    
    # 6 - Get details of recommended songs
    recommendations = df_selected.iloc[recommended_indices][
        ['track_id','name','artist','year','genre','tempo','tags','danceability','spotify_preview_url','duration_ms']
    ]
       
    
    # 7 - Add similarity score ("accuracy") column and calculation 
    input_song = input_song.assign(accuracy=["100%"])
    similarities = calculate_accuracy_from_distance(recommended_distances)
    recommendations = recommendations.assign(
        accuracy=[f"{s:.2f}%" for s in similarities]
    )
    
    # 8 - Merge input + recommendations into one DataFrame
    result = pd.concat([input_song, recommendations]).reset_index(drop=True)
    
    # 9 - convert time ms to min
    result['duration'] = result['duration_ms'].apply(ms_to_minutes_seconds)
    result = result.drop(columns=['duration_ms'])
    
    return result

### Recommendation function using genre, tags, tempo and danceability

##### Calculation accuracy function

In [19]:
def calculate_accuracy(song_tempo, song_dance, input_tempo, input_dance):
    
    # Calculate similarity with input manually (tempo + danceability diff)
    tempo_diff = abs(song_tempo - input_tempo) / max(input_tempo, 1)
    dance_diff = abs(song_dance - input_dance)
    
    # Simple similarity formula: 1 - take 50% weight from tempo difference + 50% from danceability difference.
    sim = (1 - (0.5 * tempo_diff + 0.5 * dance_diff)) * 100
    
    # Clip the score between 0 and 100 to avoid negative or >100 values.
    sim = max(0, min(100, sim)) 
    
    return f"{sim:.2f}%"

##### Validation for tempo, danceability and year

In [20]:
def validate_inputs(song_tempo, danceability_value, year):
    
    # Validate tempo (must be numeric)
    try:
        song_tempo = float(song_tempo)
    except ValueError:
        return "Tempo must be a number (BPM)."

    # Validate danceability (0.0–1.0)
    try:
        danceability_value = float(danceability_value)
    except ValueError:
        return "Danceability must be between 0.0 and 1.0."
    if not (0.0 <= danceability_value <= 1.0):
        return "Danceability must be between 0.0 and 1.0."

    # Validate year (must be integer)
    try:
        year = int(year)
    except ValueError:
        return "Year must be an integer."

    return song_tempo, danceability_value, year


##### Main Recommendation using genre, tags, tempo and danceability Function

In [21]:
def recommend_songs_by_genre_tempo_tags_danceability(
    song_genre, song_tempo, song_tags, danceability_value, year, n_recommendations=5
):
    # Normalize text fields (remove spaces + lowercase for consistency)
    song_genre = song_genre.lower().strip()
    song_tags = song_tags.lower().strip()

    # Run input validation
    validated = validate_inputs(song_tempo, danceability_value, year)
    
    # error message returned
    if isinstance(validated, str):  
        return validated 
    song_tempo, danceability_value, year = validated

    # 1 - fallback search
    candidates = None
    key_tag = (song_tags, song_genre, year)
    
    # First try - (tag + genre + year) exact match
    if key_tag in tag_genre_year_to_indices:
        candidates = df_selected.iloc[tag_genre_year_to_indices[key_tag]]

    # Second try - (genre + year)
    elif (song_genre, year) in genre_year_to_indices:
        candidates = df_selected.iloc[genre_year_to_indices[(song_genre, year)]]

    # Third try - only genre
    elif song_genre in genre_to_indices:
        candidates = df_selected.iloc[genre_to_indices[song_genre]]

    # If nothing matches - return not found message
    if candidates is None or candidates.empty:
        return f"No songs found in genre '{song_genre}' for year {year} (with/without tag '{song_tags}')."

    # 2 - Tempo filter ±5
    tempo_matches = candidates[(candidates['tempo'] >= song_tempo - 5) & (candidates['tempo'] <= song_tempo + 5)]
    if tempo_matches.empty:
        return f"No songs in genre '{song_genre}' near {song_tempo} BPM."

    # 3 - Danceability filter ±0.1
    final_matches = tempo_matches[
        (tempo_matches['danceability'] >= danceability_value - 0.1) & 
        (tempo_matches['danceability'] <= danceability_value + 0.1)
    ].copy()
    if final_matches.empty:
        return f"No songs in genre '{song_genre}' near {song_tempo} BPM with danceability {danceability_value}."

    # 4 - Pick the "closest" candidate (min absolute difference in danceability)
    final_matches['diff'] = (
        (final_matches['tempo'] - song_tempo).abs() +
        (final_matches['danceability'] - danceability_value).abs() + 
        (final_matches['year'] - year).abs()
    )
    idx = final_matches['diff'].idxmin()

    # 5 - Run KNN
    distances,indices = knn_model.kneighbors(combined_matrix[idx], n_neighbors=n_recommendations + 1)
    recommended_indices = indices.flatten()[1:]
    # recommended_distances = distances.flatten()[1:]

    # 6 - Get details of the input song
    input_song = df_selected.iloc[[idx]][['track_id','name','artist','year','genre','tempo','tags','danceability','spotify_preview_url','duration_ms']]
    
    # Check if input exactly matches the chosen candidate (strict match on 4 fields)
    is_exact_match = (
        input_song['genre'].iloc[0].lower().strip() == song_genre and
        input_song['tags'].iloc[0].lower().strip() == song_tags and
        input_song['year'].iloc[0] == year and
        abs(input_song['tempo'].iloc[0] - song_tempo) < 1e-6 and
        abs(input_song['danceability'].iloc[0] - danceability_value) < 1e-6
    )
    
    # 7 - Get details of recommended songs
    recommendations = df_selected.iloc[recommended_indices][['track_id','name','artist','year','genre','tempo','tags','danceability','spotify_preview_url','duration_ms']]

    # 8 - Add similarity score ("accuracy") column and calculation 
    if is_exact_match:
        input_song = input_song.assign(accuracy=["100%"])
    else:
        ## Input song accuracy     
        acc = calculate_accuracy(
            input_song['tempo'].iloc[0],
            input_song['danceability'].iloc[0],
            song_tempo,
            danceability_value
        )
        input_song = input_song.assign(accuracy=[acc])
        
    ## Recommended songs accuracy
    # For each recommended song, calculate similarity to the user’s input values.
    recommendations = df_selected.iloc[recommended_indices][['track_id','name','artist','year','genre','tempo','tags','danceability','spotify_preview_url','duration_ms']].copy()
    recommendations['accuracy_val'] = recommendations.apply(
        lambda row: float(calculate_accuracy(row['tempo'], row['danceability'], song_tempo, danceability_value).replace("%","")),
        axis=1
    )
    
    # Sort by accuracy high → low
    recommendations = recommendations.sort_values(by="accuracy_val", ascending=False)
    
    # Format accuracy
    recommendations['accuracy'] = recommendations['accuracy_val'].map(lambda s: f"{s:.2f}%")
    recommendations = recommendations.drop(columns="accuracy_val")

    # 9 - Merge input + recommendations into one DataFrame
    result =  pd.concat([input_song, recommendations]).reset_index(drop=True)
    
    # 10 - convert time ms to min
    result['duration'] = result['duration_ms'].apply(ms_to_minutes_seconds)
    result = result.drop(columns=['duration_ms'])
    
    return result


# Collaborative Filtering System

## Then load the datasets 

In [22]:
user_history = pd.read_csv('data/newUserListeningHistory.csv')
music_info = pd.read_csv('data/newMusicInfo.csv')

## encode user and track id

In [23]:
user_encoder = LabelEncoder()
track_encoder = LabelEncoder()
user_history['user_index'] = user_encoder.fit_transform(user_history['user_id'])
user_history['track_index'] = track_encoder.fit_transform(user_history['track_id'])

## create sparse item matrix

In [24]:
rows = user_history['user_index']
cols = user_history['track_index']
data = user_history['playcount'].astype(np.float32)
num_users = rows.nunique()
num_items = cols.nunique()
user_item_matrix = coo_matrix((data, (rows, cols)), shape=(num_users, num_items)).tocsr()

## apply truncated SVD

In [25]:
svd = TruncatedSVD(n_components=20, random_state=42)
svd_matrix = svd.fit_transform(user_item_matrix)

## convert index to id to map them

In [26]:
track_id_to_info = music_info.set_index('track_id')[['name', 'artist']].to_dict('index')
user_index_to_id = dict(enumerate(user_encoder.classes_))
track_index_to_id = dict(enumerate(track_encoder.classes_))

## recommend based on user ID then compare for similarity then suggest song that havent exist in the album of the user input id

In [27]:
def recommend_by_user_id(user_id_input, top_n=5):
    if user_id_input not in user_encoder.classes_:
        suggestions = get_close_matches(user_id_input, user_encoder.classes_, n=3)
        return f"User ID not found. Did you mean: {', '.join(suggestions)}?"

    user_index = user_encoder.transform([user_id_input])[0]
    target_vector = svd_matrix[user_index]
    
    # Calculate cosine similarity with ALL users
    similarities = cosine_similarity([target_vector], svd_matrix)[0]

    # Get top 5 similar users (excluding self)
    top_user_indices = np.argsort(-similarities)[1:6]
    
    print(f"\nInput User ID: {user_id_input}")
    print("Top 5 similar users and their similarity scores:")
    user_similarities = {}
    for i, user_idx in enumerate(top_user_indices, 1):
        sim_id = user_index_to_id[user_idx]
        sim_score = similarities[user_idx]
        user_similarities[user_idx] = sim_score
        print(f" {i}. {sim_id} (Similarity: {sim_score:.4f})")

    print(f"\nAnalyzing songs from these {len(top_user_indices)} similar users...")
    
    # Track which users listened to which songs
    track_to_user_similarities = defaultdict(list)
    
    for sim_idx in top_user_indices:
        uid = user_index_to_id[sim_idx]
        sim_score = similarities[sim_idx]
        user_tracks = user_history[user_history['user_index'] == sim_idx]['track_index'].unique()
        
        for tidx in user_tracks:
            track_to_user_similarities[tidx].append({
                'user_id': uid,
                'user_index': sim_idx,
                'similarity': sim_score
            })

    # Get songs user has already listened to
    listened = set(user_history[user_history['user_index'] == user_index]['track_index'].unique())
    
    # Score tracks based on weighted average of similar users' preferences
    track_scores = {}
    for track_idx, user_list in track_to_user_similarities.items():
        if track_idx not in listened:  # Only recommend unheard songs
            # Weight by similarity scores
            weighted_score = sum(user_info['similarity'] for user_info in user_list)
            track_scores[track_idx] = {
                'score': weighted_score,
                'contributing_users': user_list
            }
    
    # Sort by weighted score and get top N
    top_tracks = sorted(track_scores.items(), key=lambda x: x[1]['score'], reverse=True)[:top_n]
    
    recommendations = []
    for track_idx, track_data in top_tracks:
        track_id = track_index_to_id[track_idx]
        info = track_id_to_info.get(track_id, {'name': 'Unknown', 'artist': 'Unknown'})
        
        # Format contributing users with their similarities
        contributing_users = []
        for user_info in track_data['contributing_users']:
            contributing_users.append(f"{user_info['user_id']} ({user_info['similarity']:.4f})")

         # Get Spotify URL and duration from original data
        track_row = df[df['track_id'] == track_id]
        spotify_url = track_row['spotify_preview_url'].values[0] if not track_row.empty else 'Unknown'
        duration_ms = track_row['duration_ms'].values[0] if not track_row.empty else 0
        
        # Convert duration from ms to mm:ss format
        duration_formatted = f"{int(duration_ms)//60000}:{(int(duration_ms)%60000)//1000:02d}" if duration_ms else '0:00'
        
        recommendations.append({
            'track_id': track_id,
            'name': info['name'],
            'artist': info['artist'],
            'weighted_score': track_data['score'],
            'num_similar_users': len(track_data['contributing_users']),
            'contributing_users_with_similarity': '; '.join(contributing_users),  
            'spotify_preview_url': spotify_url,  
            'duration': duration_formatted  
        })
    
    return pd.DataFrame(recommendations)

# Hybrid Recommendation System

In [28]:
# Hybrid recommendation function (combines content-based and collaborative filtering)
def hybrid_recommendation(song_name, user_id, n_recommendations=5):
    """
    Hybrid recommendation that combines:
    1. Content-based filtering based on the searched song
    2. Collaborative filtering based on user preferences
    """
    # Get content-based recommendations
    content_recs = recommend_songs_by_name(song_name, n_recommendations * 2)
    
    # Get collaborative recommendations
    collab_recs = recommend_by_user_id(user_id, n_recommendations * 2)
    
    # Combine and deduplicate (simple hybrid approach)
    # In a real implementation, you would use a more sophisticated blending method
    hybrid_results = pd.concat([content_recs, collab_recs]).drop_duplicates(subset=['track_id'])
    
    # Return top n recommendations
    return hybrid_results.head(n_recommendations)

# UI and Hybrid Recommendation System

In [29]:
# Modern Music Recommendation System UI - Copy this in sections to avoid errors

import sys
from PyQt5.QtWidgets import *
from PyQt5.QtCore import Qt, QUrl
from PyQt5.QtGui import QFont
from PyQt5.QtMultimedia import QMediaPlayer, QMediaContent
import pandas as pd

# Global variable to track currently playing song
current_playing_widget = None

class SongWidget(QWidget):
    def __init__(self, name, artist, preview_url, duration, accuracy=None, parent=None):
        super().__init__(parent)
        self.preview_url = preview_url
        self.is_playing = False
        self.media_player = QMediaPlayer()
        
        self.setStyleSheet("""
            SongWidget {
                background: qlineargradient(x1:0, y1:0, x2:0, y2:1, 
                    stop:0 #ffffff, stop:1 #f8f9fa);
                border: 1px solid #e9ecef;
                border-radius: 12px;
                margin: 4px;
            }
            SongWidget:hover {
                background: qlineargradient(x1:0, y1:0, x2:0, y2:1, 
                    stop:0 #f8f9fa, stop:1 #e9ecef);
                border: 1px solid #6c757d;
            }
        """)
        
        layout = QHBoxLayout()
        layout.setContentsMargins(16, 12, 16, 12)
        layout.setSpacing(16)
        
        # Album art placeholder
        disc_label = QLabel()
        disc_label.setFixedSize(60, 60)
        disc_label.setStyleSheet("""
            QLabel {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #667eea, stop:1 #764ba2);
                border-radius: 30px;
                color: white;
                font-size: 24px;
                font-weight: bold;
            }
        """)
        disc_label.setAlignment(Qt.AlignCenter)
        disc_label.setText("♪")
        layout.addWidget(disc_label)
        
        # Song info
        info_layout = QVBoxLayout()
        info_layout.setSpacing(4)
        
        name_label = QLabel(name)
        name_label.setStyleSheet("""
            QLabel {
                font-family: 'Segoe UI', Arial, sans-serif;
                font-size: 16px;
                font-weight: 600;
                color: #212529;
                border: none;
                padding: 0;
            }
        """)
        
        artist_label = QLabel(artist)
        artist_label.setStyleSheet("""
            QLabel {
                font-family: 'Segoe UI', Arial, sans-serif;
                font-size: 14px;
                color: #6c757d;
                border: none;
                padding: 0;
            }
        """)
        
        info_layout.addWidget(name_label)
        info_layout.addWidget(artist_label)
        info_layout.addStretch()
        layout.addLayout(info_layout)
        
        # Accuracy badge
        if accuracy:
            accuracy_label = QLabel(accuracy)
            accuracy_label.setStyleSheet("""
                QLabel {
                    background-color: #28a745;
                    color: white;
                    border-radius: 12px;
                    padding: 4px 12px;
                    font-size: 12px;
                    font-weight: 600;
                }
            """)
            layout.addWidget(accuracy_label)
        
        # Controls
        controls_layout = QVBoxLayout()
        controls_layout.setAlignment(Qt.AlignCenter)
        
        duration_label = QLabel(duration)
        duration_label.setStyleSheet("""
            QLabel {
                font-size: 12px;
                color: #6c757d;
                border: none;
            }
        """)
        duration_label.setAlignment(Qt.AlignCenter)
        
        self.play_btn = QPushButton("▶")
        self.play_btn.setFixedSize(44, 44)
        self.play_btn.setStyleSheet("""
            QPushButton {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #667eea, stop:1 #764ba2);
                color: white;
                border: none;
                border-radius: 22px;
                font-size: 16px;
            }
            QPushButton:hover {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #5a6fd8, stop:1 #6a4190);
            }
        """)
        self.play_btn.clicked.connect(self.toggle_play)
        
        controls_layout.addWidget(duration_label)
        controls_layout.addWidget(self.play_btn)
        layout.addLayout(controls_layout)
        
        self.setLayout(layout)
        self.setFixedHeight(84)
    
    def toggle_play(self):
        global current_playing_widget
        
        if current_playing_widget and current_playing_widget != self:
            current_playing_widget.stop_playback()
        
        if self.is_playing:
            self.stop_playback()
        else:
            if self.preview_url:
                self.media_player.setMedia(QMediaContent(QUrl(self.preview_url)))
                self.media_player.play()
                self.play_btn.setText("⏸")
                self.is_playing = True
                current_playing_widget = self
    
    def stop_playback(self):
        self.media_player.stop()
        self.play_btn.setText("▶")
        self.is_playing = False

class LoginPage(QWidget):
    def __init__(self, parent=None):
        super().__init__(parent)
        self.parent = parent
        self.initUI()
        
    def initUI(self):
        main_layout = QVBoxLayout()
        main_layout.setAlignment(Qt.AlignCenter)
        
        # Increase the card size to make more room for the input field
        card = QWidget()
        card.setFixedSize(450, 350)  # Increased from 400, 300
        card.setStyleSheet("""
            QWidget {
                background: white;
                border-radius: 16px;
                border: 1px solid #e9ecef;
            }
        """)
        
        # Increase the margins to give more space
        card_layout = QVBoxLayout(card)
        card_layout.setContentsMargins(40, 40, 40, 40)
        card_layout.setSpacing(20)
        
        title = QLabel("Welcome Back")
        title.setStyleSheet("""
            QLabel {
                font-size: 28px;
                font-weight: 700;
                color: #212529;
                border: none;
            }
        """)
        title.setAlignment(Qt.AlignCenter)
        card_layout.addWidget(title)
        
        subtitle = QLabel("Sign in to get personalized recommendations")
        subtitle.setStyleSheet("""
            QLabel {
                font-size: 14px;
                color: #6c757d;
                border: none;
            }
        """)
        subtitle.setAlignment(Qt.AlignCenter)
        card_layout.addWidget(subtitle)
        
        self.user_id_input = QLineEdit()
        self.user_id_input.setPlaceholderText("Enter your User ID")
        # Increase the padding and font size for the input field
        self.user_id_input.setStyleSheet("""
            QLineEdit {
                font-size: 16px;
                padding: 16px 20px;  # Increased padding
                border: 2px solid #e9ecef;
                border-radius: 8px;
                background-color: #f8f9fa;
                min-height: 25px;  # Added minimum height
            }
            QLineEdit:focus {
                border: 2px solid #667eea;
                background-color: white;
            }
        """)
        self.user_id_input.returnPressed.connect(self.login)
        card_layout.addWidget(self.user_id_input)
        
        login_btn = QPushButton("Sign In")
        login_btn.setFixedHeight(48)
        login_btn.setStyleSheet("""
            QPushButton {
                font-size: 16px;
                font-weight: 600;
                color: white;
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #667eea, stop:1 #764ba2);
                border: none;
                border-radius: 8px;
            }
            QPushButton:hover {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #5a6fd8, stop:1 #6a4190);
            }
        """)
        login_btn.clicked.connect(self.login)
        card_layout.addWidget(login_btn)
        
        back_btn = QPushButton("← Back to Home")
        back_btn.setFixedHeight(40)
        back_btn.setStyleSheet("""
            QPushButton {
                color: #6c757d;
                background: transparent;
                border: 1px solid #dee2e6;
                border-radius: 8px;
            }
            QPushButton:hover {
                background-color: #f8f9fa;
            }
        """)
        back_btn.clicked.connect(lambda: self.parent.stacked_widget.setCurrentIndex(0))
        card_layout.addWidget(back_btn)
        
        main_layout.addWidget(card)
        self.setLayout(main_layout)
        
        self.setStyleSheet("""
            LoginPage {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #667eea, stop:1 #764ba2);
            }
        """)
    
    def login(self):
        user_id = self.user_id_input.text().strip()
        if user_id:
            self.parent.current_user = user_id
            self.parent.update_recommendations()
            self.parent.stacked_widget.setCurrentIndex(0)

            self.parent.home_page.update_auth_state()
            
            QMessageBox.information(self, "Success", f"Welcome back, {user_id}!")
        else:
            QMessageBox.warning(self, "Error", "Please enter a valid User ID")
class FilterPage(QWidget):
    def __init__(self, parent=None):
        super().__init__(parent)
        self.parent = parent
        self.initUI()
        
    def initUI(self):
        main_layout = QVBoxLayout()
        main_layout.setContentsMargins(24, 24, 24, 24)
        main_layout.setSpacing(24)
        
        # Header
        header_layout = QHBoxLayout()
        
        title = QLabel("Advanced Filters")
        title.setStyleSheet("""
            QLabel {
                font-size: 32px;
                font-weight: 700;
                color: #212529;
                border: none;
            }
        """)
        
        back_btn = QPushButton("← Back")
        back_btn.setFixedSize(100, 40)
        back_btn.clicked.connect(lambda: self.parent.stacked_widget.setCurrentIndex(0))
        
        header_layout.addWidget(title)
        header_layout.addStretch()
        header_layout.addWidget(back_btn)
        main_layout.addLayout(header_layout)
        
        # Form card
        form_card = QWidget()
        form_card.setStyleSheet("""
            QWidget {
                background: white;
                border-radius: 16px;
                border: 1px solid #e9ecef;
            }
        """)
        
        form_layout = QVBoxLayout(form_card)
        form_layout.setContentsMargins(32, 32, 32, 32)
        form_layout.setSpacing(24)
        
        # Grid layout
        grid_layout = QHBoxLayout()
        
        # Left column
        left_column = QVBoxLayout()
        left_column.setSpacing(16)
        
        # Genre
        genre_label = QLabel("Genre")
        genre_label.setStyleSheet("""
            QLabel {
                font-size: 16px;
                font-weight: 600;
                color: #495057;
                border: none;
            }
        """)
        
        self.genre_combo = QComboBox()
        self.genre_combo.setStyleSheet("""
            QComboBox {
                font-size: 14px;
                padding: 12px 16px;
                border: 2px solid #e9ecef;
                border-radius: 8px;
                background-color: #f8f9fa;
                min-width: 200px;
            }
        """)
        genres = ['RNB', 'ROCK', 'POP', 'METAL', 'ELECTRONIC', 'JAZZ', 'PUNK', 
                 'COUNTRY', 'FOLK', 'REGGAE', 'RAP', 'BLUES', 'NEW AGE', 'LATIN', 'WORLD']
        self.genre_combo.addItems(genres)
        
        left_column.addWidget(genre_label)
        left_column.addWidget(self.genre_combo)
        
        # Tags
        tags_label = QLabel("Tags")
        tags_label.setStyleSheet(genre_label.styleSheet())
        
        self.tags_combo = QComboBox()
        self.tags_combo.setStyleSheet(self.genre_combo.styleSheet())
        
        tags = ['BEAUTIFUL', 'PROGRESSIVE_METAL', 'EMO', 'DANCE', 'PSYCHEDELIC', 
                'GOTHIC_METAL', 'DOWNTEMPO', 'POWER_METAL', 'SYNTHPOP', 'DEATH_METAL']
        self.tags_combo.addItems(tags)
        
        left_column.addWidget(tags_label)
        left_column.addWidget(self.tags_combo)
        
        grid_layout.addLayout(left_column)
        
        # Right column
        right_column = QVBoxLayout()
        right_column.setSpacing(16)
        
        # Year
        year_label = QLabel("Year")
        year_label.setStyleSheet(genre_label.styleSheet())
        
        self.year_input = QSpinBox()
        self.year_input.setStyleSheet("""
            QSpinBox {
                font-size: 14px;
                padding: 12px 16px;
                border: 2px solid #e9ecef;
                border-radius: 8px;
                background-color: #f8f9fa;
                min-width: 200px;
            }
        """)
        self.year_input.setRange(1900, 2030)
        self.year_input.setValue(2020)
        
        right_column.addWidget(year_label)
        right_column.addWidget(self.year_input)
        
        # Tempo
        tempo_label = QLabel("Tempo (BPM)")
        tempo_label.setStyleSheet(genre_label.styleSheet())
        
        self.tempo_input = QSpinBox()
        self.tempo_input.setStyleSheet(self.year_input.styleSheet())
        self.tempo_input.setRange(0, 239)
        self.tempo_input.setValue(120)
        
        right_column.addWidget(tempo_label)
        right_column.addWidget(self.tempo_input)
        
        # Danceability
        dance_label = QLabel("Danceability (0.0 - 1.0)")
        dance_label.setStyleSheet(genre_label.styleSheet())
        
        self.dance_input = QDoubleSpinBox()
        self.dance_input.setStyleSheet("""
            QDoubleSpinBox {
                font-size: 14px;
                padding: 12px 16px;
                border: 2px solid #e9ecef;
                border-radius: 8px;
                background-color: #f8f9fa;
                min-width: 200px;
            }
        """)
        self.dance_input.setRange(0.0, 1.0)
        self.dance_input.setSingleStep(0.1)
        self.dance_input.setValue(0.5)
        
        right_column.addWidget(dance_label)
        right_column.addWidget(self.dance_input)
        
        grid_layout.addLayout(right_column)
        form_layout.addLayout(grid_layout)
        
        # Filter button
        filter_btn = QPushButton("Find Matching Songs")
        filter_btn.setFixedHeight(48)
        filter_btn.setStyleSheet("""
            QPushButton {
                font-size: 16px;
                font-weight: 600;
                color: white;
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #667eea, stop:1 #764ba2);
                border: none;
                border-radius: 8px;
            }
            QPushButton:hover {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #5a6fd8, stop:1 #6a4190);
            }
        """)
        filter_btn.clicked.connect(self.apply_filters)
        form_layout.addWidget(filter_btn)
        
        main_layout.addWidget(form_card)
        
        # Results
        results_label = QLabel("Matching Songs")
        results_label.setStyleSheet("""
            QLabel {
                font-size: 20px;
                font-weight: 600;
                color: #495057;
                border: none;
            }
        """)
        main_layout.addWidget(results_label)
        
        self.results_list = QListWidget()
        self.results_list.setStyleSheet("""
            QListWidget {
                border: none;
                background: transparent;
            }
            QListWidget::item {
                border: none;
                padding: 4px 0;
                background: transparent;
            }
        """)
        main_layout.addWidget(self.results_list)
        
        self.setLayout(main_layout)
        self.setStyleSheet("FilterPage { background-color: #f8f9fa; }")
    
    def apply_filters(self):
        selected_tag = self.tags_combo.currentText()
        
        recommendations = recommend_songs_by_genre_tempo_tags_danceability(
            song_genre=self.genre_combo.currentText(),
            song_tempo=str(self.tempo_input.value()),
            song_tags=selected_tag,
            danceability_value=str(self.dance_input.value()),
            year=str(self.year_input.value())
        )
        
        self.results_list.clear()
        if isinstance(recommendations, str):
            error_item = QListWidgetItem()
            error_widget = QLabel(recommendations)
            error_widget.setStyleSheet("""
                QLabel {
                    color: #dc3545;
                    background: #f8d7da;
                    border: 1px solid #f5c6cb;
                    border-radius: 8px;
                    padding: 12px;
                }
            """)
            error_item.setSizeHint(error_widget.sizeHint())
            self.results_list.addItem(error_item)
            self.results_list.setItemWidget(error_item, error_widget)
        else:
            for _, row in recommendations.iterrows():
                list_item = QListWidgetItem()
                song_widget = SongWidget(
                    row['name'], 
                    row['artist'], 
                    row['spotify_preview_url'],
                    row['duration'],
                    row.get('accuracy', None)
                )
                list_item.setSizeHint(song_widget.sizeHint())
                self.results_list.addItem(list_item)
                self.results_list.setItemWidget(list_item, song_widget)

class HomePage(QWidget):
    def __init__(self, parent=None):
        super().__init__(parent)
        self.parent = parent
        self.initUI()
        
    def initUI(self):
        main_layout = QVBoxLayout()
        main_layout.setContentsMargins(24, 24, 24, 24)
        main_layout.setSpacing(24)
        
        # Header
        header_card = QWidget()
        header_card.setStyleSheet("""
            QWidget {
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #667eea, stop:1 #764ba2);
                border-radius: 16px;
            }
        """)
        header_card.setFixedHeight(120)
        
        header_layout = QVBoxLayout(header_card)
        header_layout.setContentsMargins(32, 24, 32, 24)
        header_layout.setSpacing(16)
        
        title = QLabel("Music Discovery")
        title.setStyleSheet("""
            QLabel {
                font-size: 28px;
                font-weight: 700;
                color: white;
                border: none;
            }
        """)
        header_layout.addWidget(title)
        
        # Search bar
        search_layout = QHBoxLayout()
        
        self.search_input = QLineEdit()
        self.search_input.setPlaceholderText("Search for a song...")
        self.search_input.setStyleSheet("""
            QLineEdit {
                color: black;                      /* Make the text visible */
                background-color: white;           /* Set a solid background */
                border: 2px solid #A66CFF;         /* Match your gradient theme */
                padding: 6px;
                font-size: 14px;
                border-radius: 6px;
            }
            QLineEdit:focus {
                border: 2px solid #7B2FF7;         /* Highlight on focus */
            }
        """)
        self.search_input.returnPressed.connect(self.search_songs)
        
        search_btn = QPushButton("Search")
        search_btn.setFixedSize(80, 50)
        search_btn.setStyleSheet("""
            QPushButton {
                background: rgba(255, 255, 255, 0.2);
                border: 2px solid rgba(255, 255, 255, 0.3);
                border-radius: 25px;
                color: white;
                font-weight: 600;
            }
            QPushButton:hover {
                background: rgba(255, 255, 255, 0.3);
            }
        """)
        search_btn.clicked.connect(self.search_songs)
        
        search_layout.addWidget(self.search_input)
        search_layout.addWidget(search_btn)
        header_layout.addLayout(search_layout)
        
        main_layout.addWidget(header_card)
        
        # Navigation
        self.nav_layout = QHBoxLayout()
        
        filter_btn = QPushButton("Advanced Filters")
        filter_btn.setFixedHeight(44)
        filter_btn.setStyleSheet("""
            QPushButton {
                color: #495057;
                background: white;
                border: 1px solid #dee2e6;
                border-radius: 22px;
                padding: 8px 20px;
            }
            QPushButton:hover {
                background: #f8f9fa;
            }
        """)
        filter_btn.clicked.connect(lambda: self.parent.stacked_widget.setCurrentIndex(2))
        
        # Create login/logout button
        self.auth_btn = QPushButton("Sign In")
        self.auth_btn.setFixedHeight(44)
        self.auth_btn.setStyleSheet(filter_btn.styleSheet())
        self.auth_btn.clicked.connect(self.toggle_auth)
        
        self.nav_layout.addWidget(filter_btn)
        self.nav_layout.addStretch()
        self.nav_layout.addWidget(self.auth_btn)
        
        main_layout.addLayout(self.nav_layout)
        
       # Content (3 portrait columns)
        content_scroll = QScrollArea()
        content_scroll.setWidgetResizable(True)
        content_scroll.setStyleSheet("""
            QScrollArea {
                border: none;
                background: transparent;
            }
        """)
        
        content_widget = QWidget()
        row = QHBoxLayout(content_widget)
        row.setSpacing(16)
        row.setContentsMargins(0, 0, 0, 0)
        
        def build_portrait(title_text):
            card = QWidget()
            card.setStyleSheet("""
                QWidget {
                    background: white;
                    border-radius: 16px;
                    border: 1px solid #e9ecef;
                }
            """)
            card_layout = QVBoxLayout(card)
            card_layout.setContentsMargins(16, 16, 16, 16)
            card_layout.setSpacing(12)
        
            title = QLabel(title_text)
            title.setStyleSheet("""
                QLabel {
                    font-size: 18px;
                    font-weight: 600;
                    color: #495057;
                    border: none;
                }
            """)
            card_layout.addWidget(title)
        
            lst = QListWidget()
            lst.setStyleSheet("""
                QListWidget {
                    border: none;
                    background: transparent;
                }
                QListWidget::item {
                    border: none;
                    padding: 4px 0;
                    background: transparent;
                }
            """)
            lst.setSizePolicy(QSizePolicy.Preferred, QSizePolicy.Expanding)
            card_layout.addWidget(lst, 1)
        
            # make each card behave like a portrait column
            card.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
            return card, lst
        
        # Build three portraits
        self.search_panel, self.search_results_list = build_portrait("Search Results")
        self.hybrid_panel, self.hybrid_list = build_portrait("Hybrid Recommendations")
        self.user_panel, self.user_rec_list = build_portrait("Recommended for You")
        
        # Start with hybrid panel hidden until we actually have results
        self.hybrid_panel.hide()
        
        row.addWidget(self.search_panel, 1)
        row.addWidget(self.hybrid_panel, 1)
        row.addWidget(self.user_panel, 1)
        
        # Welcome message (initially hidden) — keep as is
        self.welcome_label = QLabel()
        self.welcome_label.setStyleSheet("""
            QLabel {
                font-size: 16px;
                color: #495057;
                background: white;
                border-radius: 8px;
                padding: 12px;
                border: 1px solid #e9ecef;
            }
        """)
        self.welcome_label.hide()

# Put welcome at the top of the left portrait
# (so it doesn't push layout elsewhere)
        left_panel_layout = self.search_panel.layout()
        left_panel_layout.insertWidget(1, self.welcome_label)
        
        content_scroll.setWidget(content_widget)
        main_layout.addWidget(content_scroll)

        
        self.setLayout(main_layout)
        self.setStyleSheet("HomePage { background-color: #f8f9fa; }")
        
        # Update UI based on initial login state
        self.update_auth_state()
    
    def toggle_auth(self):
        if self.parent.current_user:
            # Logout
            self.parent.current_user = None
            QMessageBox.information(self, "Logged Out", "You have been successfully logged out.")
        else:
            # Login
            self.parent.stacked_widget.setCurrentIndex(1)
        
        self.update_auth_state()
    
    def update_auth_state(self):
        if self.parent.current_user:
            # User is logged in
            self.auth_btn.setText("Logout")
            self.welcome_label.setText(f"Welcome, {self.parent.current_user}! Enjoy personalized recommendations.")
            self.welcome_label.show()
            
            # Remove login prompt if it exists
            self.clear_login_prompt()
            
            # Update recommendations
            self.parent.update_recommendations()
        else:
            # User is not logged in
            self.auth_btn.setText("Sign In")
            self.welcome_label.hide()
            
            # Show login prompt
            self.show_login_prompt()
    
    def clear_login_prompt(self):
        # Remove any existing login prompt
        for i in range(self.user_rec_list.count()):
            item = self.user_rec_list.item(i)
            widget = self.user_rec_list.itemWidget(item)
            if widget and hasattr(widget, 'property') and widget.property('login_prompt'):
                self.user_rec_list.takeItem(i)
                break
    
    def show_login_prompt(self):
        self.clear_login_prompt()  # Clear any existing prompt first
        
        login_prompt = QWidget()
        login_prompt.setProperty('login_prompt', True)
        login_prompt.setStyleSheet("""
            QWidget {
                background: white;
                border-radius: 12px;
                border: 1px solid #e9ecef;
            }
        """)
        login_prompt_layout = QVBoxLayout(login_prompt)
        login_prompt_layout.setContentsMargins(24, 20, 24, 20)
        
        prompt_title = QLabel("Get Personalized Recommendations")
        prompt_title.setStyleSheet("""
            QLabel {
                font-size: 16px;
                font-weight: 600;
                color: #495057;
                border: none;
            }
        """)
        
        prompt_text = QLabel("Sign in to discover music tailored just for you.")
        prompt_text.setStyleSheet("""
            QLabel {
                font-size: 14px;
                color: #6c757d;
                border: none;
            }
        """)
        prompt_text.setWordWrap(True)
        
        prompt_login_btn = QPushButton("Sign In Now")
        prompt_login_btn.setFixedHeight(40)
        prompt_login_btn.setStyleSheet("""
            QPushButton {
                font-weight: 600;
                color: white;
                background: qlineargradient(x1:0, y1:0, x2:1, y2:1,
                    stop:0 #667eea, stop:1 #764ba2);
                border: none;
                border-radius: 8px;
            }
        """)
        prompt_login_btn.clicked.connect(lambda: self.parent.stacked_widget.setCurrentIndex(1))
        
        login_prompt_layout.addWidget(prompt_title)
        login_prompt_layout.addWidget(prompt_text)
        login_prompt_layout.addWidget(prompt_login_btn)
        
        list_item = QListWidgetItem()
        list_item.setSizeHint(login_prompt.sizeHint())
        self.user_rec_list.insertItem(0, list_item)
        self.user_rec_list.setItemWidget(list_item, login_prompt)
    
    def search_songs(self):
        song_name = self.search_input.text().strip()
        if song_name:
            recommendations = recommend_songs_by_name(song_name)
            self.search_results_list.clear()
            
            if isinstance(recommendations, str):
                error_item = QListWidgetItem()
                error_widget = QLabel(recommendations)
                error_widget.setStyleSheet("""
                    QLabel {
                        color: #dc3545;
                        background: #f8d7da;
                        border: 1px solid #f5c6cb;
                        border-radius: 8px;
                        padding: 12px;
                    }
                """)
                error_item.setSizeHint(error_widget.sizeHint())
                self.search_results_list.addItem(error_item)
                self.search_results_list.setItemWidget(error_item, error_widget)
            else:
                for _, row in recommendations.iterrows():
                    list_item = QListWidgetItem()
                    song_widget = SongWidget(
                        row['name'], 
                        row['artist'], 
                        row['spotify_preview_url'],
                        row['duration'],
                        row.get('accuracy', None)
                    )
                    list_item.setSizeHint(song_widget.sizeHint())
                    self.search_results_list.addItem(list_item)
                    self.search_results_list.setItemWidget(list_item, song_widget)
            
            # Show hybrid if logged in
            if self.parent.current_user:
                try:
                    hybrid_recs = hybrid_recommendation(song_name, self.parent.current_user)
                    self.hybrid_list.clear()
                    
                    if isinstance(hybrid_recs, pd.DataFrame) and not hybrid_recs.empty:
                        for _, row in hybrid_recs.iterrows():
                            list_item = QListWidgetItem()
                            song_widget = SongWidget(
                                row['name'], 
                                row['artist'], 
                                row['spotify_preview_url'],
                                row.get('duration', '0:00')
                            )
                            list_item.setSizeHint(song_widget.sizeHint())
                            self.hybrid_list.addItem(list_item)
                            self.hybrid_list.setItemWidget(list_item, song_widget)
                    
                    self.hybrid_panel.show()
                except Exception as e:
                    print(f"Hybrid recommendation error: {e}")
                    self.hybrid_panel.hide()
            else:
                self.hybrid_label.hide()
                self.hybrid_list.hide()
                
class MusicRecommendationApp(QMainWindow):
    def __init__(self):
        super().__init__()
        self.current_user = None
        self.initUI()
        
    def initUI(self):
        self.setWindowTitle("Music Discovery - AI Powered Recommendations")
        self.setGeometry(100, 100, 1200, 800)
        
        central_widget = QWidget()
        self.setCentralWidget(central_widget)
        
        # Stacked widget for pages
        self.stacked_widget = QStackedWidget()
        
        # Create pages
        self.home_page = HomePage(self)
        self.login_page = LoginPage(self)
        self.filter_page = FilterPage(self)
        
        # Add pages
        self.stacked_widget.addWidget(self.home_page)
        self.stacked_widget.addWidget(self.login_page)
        self.stacked_widget.addWidget(self.filter_page)
        
        # Layout
        layout = QVBoxLayout()
        layout.setContentsMargins(0, 0, 0, 0)
        layout.addWidget(self.stacked_widget)
        central_widget.setLayout(layout)
        
        self.apply_styles()
    
    def apply_styles(self):
        self.setStyleSheet("""
            QMainWindow {
                background-color: #f8f9fa;
                font-family: 'Segoe UI', Arial, sans-serif;
            }
        """)
    
    def update_recommendations(self):
        if self.current_user:
            try:
                recommendations = recommend_by_user_id(self.current_user)
                self.home_page.user_rec_list.clear()
                
                if isinstance(recommendations, pd.DataFrame) and not recommendations.empty:
                    for _, row in recommendations.iterrows():
                        list_item = QListWidgetItem()
                        song_widget = SongWidget(
                            row['name'], 
                            row['artist'], 
                            row['spotify_preview_url'],
                            row['duration']
                        )
                        list_item.setSizeHint(song_widget.sizeHint())
                        self.home_page.user_rec_list.addItem(list_item)
                        self.home_page.user_rec_list.setItemWidget(list_item, song_widget)
                else:
                    no_recs_item = QListWidgetItem()
                    no_recs_widget = QLabel("No recommendations available for this user.")
                    no_recs_widget.setStyleSheet("""
                        QLabel {
                            color: #6c757d;
                            background: white;
                            border: 1px solid #e9ecef;
                            border-radius: 8px;
                            padding: 20px;
                        }
                    """)
                    no_recs_item.setSizeHint(no_recs_widget.sizeHint())
                    self.home_page.user_rec_list.addItem(no_recs_item)
                    self.home_page.user_rec_list.setItemWidget(no_recs_item, no_recs_widget)
                    
            except Exception as e:
                print(f"Error updating recommendations: {e}")

def main():
    app = QApplication(sys.argv)
    app.setApplicationName("Music Discovery")
    app.setApplicationVersion("1.0")
    
    window = MusicRecommendationApp()
    window.show()
    sys.exit(app.exec_())

if __name__ == '__main__':
    main()

AttributeError: 'HomePage' object has no attribute 'hybrid_label'

SystemExit: 0

C:\Users\Lwin\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
